# Week 9 Coding Practice: Relational Data in Applied Data Science (Instructor Solution)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/DataScience-book/blob/w09-cp-01-colab-link/module09/week09_relational_data_practice_solution.ipynb)

**Estimated time:** 60–90 minutes
**Format:** instructor solution

This notebook contains a completed solution for the bike-share relational-data coding practice. The data are synthetic and explicitly labeled as such.


## Learning objectives

By the end of this activity, students should be able to:

1. explain the relationship between stations and trips;
2. inspect raw relational data for missing values and invalid references;
3. compare different join strategies and explain what each type of join keeps;
4. diagnose a misleading count after a left join and repair it;
5. build a station-level analysis table with one row per station;
6. validate the final table before interpreting it; and
7. use the validated station table to answer a neighborhood-level question.


In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')

candidate_paths = [
    Path('appliedDataScience/module09/stations.csv'),
    Path('module09/stations.csv'),
    Path.cwd() / 'appliedDataScience/module09/stations.csv',
]
stations_path = next((p for p in candidate_paths if p.exists()), candidate_paths[0])

trip_paths = [
    Path('appliedDataScience/module09/trips.csv'),
    Path('module09/trips.csv'),
    Path.cwd() / 'appliedDataScience/module09/trips.csv',
]
trips_path = next((p for p in trip_paths if p.exists()), trip_paths[0])

print('Stations path:', stations_path)
print('Trips path:', trips_path)

stations = pd.read_csv(stations_path)
trips = pd.read_csv(trips_path)

conn = sqlite3.connect(':memory:')
stations.to_sql('stations', conn, index=False, if_exists='replace')
trips.to_sql('trips', conn, index=False, if_exists='replace')

print('Stations rows:', len(stations))
print('Trips rows:', len(trips))
print(stations.head())
print(trips.head())


In [ ]:
query = """
SELECT
    'stations' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT station_id) AS distinct_station_ids
FROM stations

UNION ALL

SELECT
    'trips' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT trip_id) AS distinct_trip_ids
FROM trips;
"""

pd.read_sql_query(query, conn)


In [ ]:
query = """
SELECT
    COUNT(*) AS total_trips,
    SUM(CASE WHEN start_station_id IS NULL THEN 1 ELSE 0 END) AS missing_start_station_id,
    SUM(CASE WHEN start_station_id IS NOT NULL AND start_station_id NOT IN (SELECT station_id FROM stations) THEN 1 ELSE 0 END) AS unmatched_start_station_id,
    SUM(CASE WHEN start_station_id IS NOT NULL AND start_station_id IN (SELECT station_id FROM stations) THEN 1 ELSE 0 END) AS matched_station_ids
FROM trips;
"""

pd.read_sql_query(query, conn)


In [ ]:
inner_join = """
SELECT s.station_id, s.station_name, s.neighborhood, t.trip_id, t.duration_minutes
FROM stations s
INNER JOIN trips t
    ON s.station_id = t.start_station_id
ORDER BY s.station_id, t.trip_id
LIMIT 12;
"""

print('INNER JOIN sample:')
print(pd.read_sql_query(inner_join, conn).head(12).to_string(index=False))

left_join = """
SELECT s.station_id, s.station_name, s.neighborhood, t.trip_id, t.duration_minutes
FROM stations s
LEFT JOIN trips t
    ON s.station_id = t.start_station_id
ORDER BY s.station_id, t.trip_id
LIMIT 12;
"""
print('LEFT JOIN sample:')
print(pd.read_sql_query(left_join, conn).head(12).to_string(index=False))


In [ ]:
misleading_query = """
SELECT
    s.station_id,
    s.station_name,
    COUNT(*) AS reported_departures
FROM stations s
LEFT JOIN trips t
    ON s.station_id = t.start_station_id
GROUP BY s.station_id, s.station_name
ORDER BY s.station_id;
"""

print('Misleading COUNT(*) summary:')
print(pd.read_sql_query(misleading_query, conn).head(10).to_string(index=False))

corrected_query = """
SELECT
    s.station_id,
    s.station_name,
    COUNT(t.trip_id) AS recorded_departures,
    COUNT(t.duration_minutes) AS departures_with_observed_duration,
    AVG(t.duration_minutes) AS mean_duration_minutes
FROM stations s
LEFT JOIN trips t
    ON s.station_id = t.start_station_id
GROUP BY s.station_id, s.station_name
ORDER BY s.station_id;
"""

print('Corrected summary:')
print(pd.read_sql_query(corrected_query, conn).head(10).to_string(index=False))


In [ ]:
trip_summary = """
WITH trip_summary AS (
    SELECT
        start_station_id,
        COUNT(trip_id) AS recorded_departure_count,
        COUNT(duration_minutes) AS departures_with_observed_duration,
        AVG(duration_minutes) AS mean_duration_minutes
    FROM trips
    WHERE start_station_id IS NOT NULL
      AND start_station_id IN (SELECT station_id FROM stations)
    GROUP BY start_station_id
)
SELECT
    s.station_id,
    s.station_name,
    s.neighborhood,
    COALESCE(ts.recorded_departure_count, 0) AS recorded_departure_count,
    COALESCE(ts.departures_with_observed_duration, 0) AS departures_with_observed_duration,
    ts.mean_duration_minutes
FROM stations s
LEFT JOIN trip_summary ts
    ON s.station_id = ts.start_station_id
ORDER BY s.station_id;
"""

station_summary = pd.read_sql_query(trip_summary, conn)
print(station_summary.to_string(index=False))


In [ ]:
validations = """
WITH trip_summary AS (
    SELECT
        start_station_id,
        COUNT(trip_id) AS recorded_departure_count,
        COUNT(duration_minutes) AS departures_with_observed_duration,
        AVG(duration_minutes) AS mean_duration_minutes
    FROM trips
    WHERE start_station_id IS NOT NULL
      AND start_station_id IN (SELECT station_id FROM stations)
    GROUP BY start_station_id
), station_summary AS (
    SELECT
        s.station_id,
        s.station_name,
        s.neighborhood,
        COALESCE(ts.recorded_departure_count, 0) AS recorded_departure_count,
        COALESCE(ts.departures_with_observed_duration, 0) AS departures_with_observed_duration,
        ts.mean_duration_minutes
    FROM stations s
    LEFT JOIN trip_summary ts
        ON s.station_id = ts.start_station_id
), matched AS (
    SELECT COUNT(*) AS matched_trips
    FROM trips
    WHERE start_station_id IS NOT NULL
      AND start_station_id IN (SELECT station_id FROM stations)
), unmatched AS (
    SELECT COUNT(*) AS unmatched_trips
    FROM trips
    WHERE start_station_id IS NOT NULL
      AND start_station_id NOT IN (SELECT station_id FROM stations)
), missing_key AS (
    SELECT COUNT(*) AS missing_key_trips
    FROM trips
    WHERE start_station_id IS NULL
), raw_total AS (
    SELECT COUNT(*) AS total_trips FROM trips
)
SELECT
    (SELECT COUNT(*) FROM station_summary) AS station_rows,
    (SELECT COUNT(DISTINCT station_id) FROM station_summary) AS distinct_station_ids,
    (SELECT COUNT(*) FROM station_summary WHERE recorded_departure_count = 0) AS stations_with_no_recorded_departures,
    (SELECT SUM(recorded_departure_count) FROM station_summary) AS sum_of_station_departure_counts,
    (SELECT matched_trips FROM matched) AS matched_trips,
    (SELECT unmatched_trips FROM unmatched) AS unmatched_trips,
    (SELECT missing_key_trips FROM missing_key) AS missing_key_trips,
    (SELECT total_trips FROM raw_total) AS total_trips;
"""

print(pd.read_sql_query(validations, conn).to_string(index=False))


In [ ]:
station_summary = pd.read_sql_query("""
WITH trip_summary AS (
    SELECT
        start_station_id,
        COUNT(trip_id) AS recorded_departure_count,
        COUNT(duration_minutes) AS departures_with_observed_duration,
        AVG(duration_minutes) AS mean_duration_minutes
    FROM trips
    WHERE start_station_id IS NOT NULL
      AND start_station_id IN (SELECT station_id FROM stations)
    GROUP BY start_station_id
)
SELECT
    s.station_id,
    s.station_name,
    s.neighborhood,
    COALESCE(ts.recorded_departure_count, 0) AS recorded_departure_count,
    COALESCE(ts.departures_with_observed_duration, 0) AS departures_with_observed_duration,
    ts.mean_duration_minutes
FROM stations s
LEFT JOIN trip_summary ts
    ON s.station_id = ts.start_station_id
ORDER BY s.station_id;
""", conn)

station_summary = station_summary.sort_values('recorded_departure_count', ascending=False)
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(station_summary['station_name'], station_summary['recorded_departure_count'], color='#2c7fb8')
ax.set_title('Recorded departures by station')
ax.set_xlabel('Station')
ax.set_ylabel('Recorded departures')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
neighborhood_summary = (
    station_summary.groupby('neighborhood', as_index=False)
    .agg(
        station_count=('station_id', 'size'),
        total_departures=('recorded_departure_count', 'sum'),
        mean_departures_per_station=('recorded_departure_count', 'mean')
    )
    .sort_values('mean_departures_per_station', ascending=False)
)

print(neighborhood_summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(neighborhood_summary['neighborhood'], neighborhood_summary['mean_departures_per_station'], color='#7a3db8')
ax.set_title('Average recorded departures per station by neighborhood')
ax.set_xlabel('Neighborhood')
ax.set_ylabel('Average departures per station')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()
